# Let an LLM call the PointNet classifier as a tool

The LLM is the **orchestrator**, not the classifier. The user asks in plain
language; the LLM calls our trained **PointNet** model as a *tool* (function
calling); our Python runs the actual inference; the LLM phrases the answer.

**No dataset download** — this notebook only needs:
- `pointnet_modelnet10.keras` — the trained model (already saved).
- One `.off` mesh file to classify (set its path in section 5).
- A free **Groq** API key: https://console.groq.com

## 1. Install

In [ ]:
!pip install openai trimesh

## 2. Configure the LLM client

`openai` is just a thin OpenAI-compatible client — requests go to Groq.

In [ ]:
import os
from openai import OpenAI

# Read the key from the environment — never hardcode it.
# In a terminal before launching Jupyter:  export GROQ_API_KEY="your_key"
# or uncomment for a quick local test (do NOT commit it):
# os.environ["GROQ_API_KEY"] = "gsk_..."

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

LLM_MODEL = "llama-3.3-70b-versatile"  # supports tool calling

## 3. Load the model and define the tool function

`CLASS_MAP` is hardcoded (the 10 ModelNet10 classes in the same sorted order the
model was trained on), so no dataset scan is needed. The custom regularizer is
re-declared because it's required to deserialize the saved model.

In [ ]:
import numpy as np
import trimesh
import keras


# ModelNet10 classes in sorted alphabetical order (matches training label order)
CLASS_MAP = {
    0: "bathtub",
    1: "bed",
    2: "chair",
    3: "desk",
    4: "dresser",
    5: "monitor",
    6: "night_stand",
    7: "sofa",
    8: "table",
    9: "toilet",
}


# Re-declare the custom regularizer (needed to load the saved model)
@keras.saving.register_keras_serializable()
class OrthogonalRegularizer(keras.regularizers.Regularizer):
    def __init__(self, num_features, l2reg=0.001):
        self.num_features = num_features
        self.l2reg = l2reg
        self.eye = keras.ops.eye(num_features)

    def __call__(self, x):
        x = keras.ops.reshape(x, (-1, self.num_features, self.num_features))
        xxt = keras.ops.tensordot(x, x, axes=(2, 2))
        xxt = keras.ops.reshape(xxt, (-1, self.num_features, self.num_features))
        return keras.ops.sum(self.l2reg * keras.ops.square(xxt - self.eye))

    def get_config(self):
        return {"num_features": self.num_features, "l2reg": self.l2reg}


pointnet = keras.models.load_model(
    "pointnet_modelnet10.keras",
    custom_objects={"OrthogonalRegularizer": OrthogonalRegularizer},
)


def classify_point_cloud(file_path, num_points=2048):
    """Run PointNet on a single .off mesh and return the predicted class."""
    cloud = trimesh.load(file_path).sample(num_points)
    cloud = np.expand_dims(cloud.astype("float32"), axis=0)  # (1, N, 3)

    probs = pointnet.predict(cloud, verbose=0)[0]
    idx = int(np.argmax(probs))
    return {
        "predicted_class": CLASS_MAP[idx],
        "confidence": round(float(probs[idx]), 4),
    }

## 4. Describe the tool to the LLM

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "classify_point_cloud",
            "description": (
                "Classify a 3D object from its point cloud using a trained "
                "PointNet model. Returns the predicted ModelNet10 class and "
                "a confidence score."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {
                        "type": "string",
                        "description": "Path to the .off mesh file to classify.",
                    }
                },
                "required": ["file_path"],
            },
        },
    }
]

## 5. The tool-calling loop

In [ ]:
import json


def ask_llm(user_message):
    messages = [
        {
            "role": "system",
            "content": (
                "You help classify 3D objects. When the user references a point "
                "cloud file, call the classify_point_cloud tool to identify it, "
                "then explain the result in plain language."
            ),
        },
        {"role": "user", "content": user_message},
    ]

    # First call — the LLM decides whether to use the tool
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    msg = response.choices[0].message

    if not msg.tool_calls:
        return msg.content

    messages.append(msg)
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = classify_point_cloud(**args)   # <-- PointNet runs here
        print(f"[tool] classify_point_cloud({args}) -> {result}")
        messages.append(
            {
                "role": "tool",
                "tool_call_id": call.id,
                "name": call.function.name,
                "content": json.dumps(result),
            }
        )

    final = client.chat.completions.create(model=LLM_MODEL, messages=messages)
    return final.choices[0].message.content

## 6. Try it

Set `test_file` to any `.off` mesh you have.

In [ ]:
test_file = "chair_0890.off"   # <-- change to a real path on your machine

answer = ask_llm(f"What object is in this point cloud file: {test_file} ?")
print("\n" + answer)